# Clinical Evidence Research Agent Using Strands - External & Internal Tool Integration
This notebook creates a clinical evidence research agent using the open-source Strands agent framework

#### Install Strands agents and required dependencies

In [ ]:
%pip install strands-agents strands-agents-tools==0.2.9 xmltodict --quiet

#### Verify latest version of boto3 shown below
Verify that the boto3 version shown below is **1.37.1** or higher.

In [ ]:
%pip show boto3

#### Import required libraries

In [ ]:
import os
import boto3
import json
import uuid
import requests
from pathlib import Path
from typing import Dict, Any, List
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

from utils.PubMed import PubMed

# KB 도구 변수 초기화
kb_tool = None

# AWS 계정 정보 확인
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name

#### Upload documents from documents/ directory to S3 bucket

In [ ]:
# Initialize S3 client
s3_client = boto3.client('s3')
environment_name = 'env1'
bucket_name = f'clinical-kb-{environment_name}-{account_id}-{region}'

documents_dir = Path('./documents')
if documents_dir.exists():
    for file_path in documents_dir.rglob('*'):
        if file_path.is_file():
            s3_key = str(file_path.relative_to('.'))
            try:
                s3_client.upload_file(str(file_path), bucket_name, s3_key)
                print(f'Upload complete: {s3_key} -> s3://{bucket_name}/{s3_key}')
            except Exception as e:
                print(f'Upload failed: {s3_key} - {str(e)}')
else:
    print('documents/ directory does not exist.')

#### Set up AWS clients
Define AWS service clients to use in tools.

In [ ]:
# Initialize AWS clients
bedrock_client = boto3.client('bedrock-runtime', region_name=region)
bedrock_agent_client = boto3.client("bedrock-agent", region_name=region)

print(f"Region: {region}")
print(f"Account ID: {account_id}")

#### Set up knowledge base for internal document search

In this example, we will use the built-in tool `retrieve` from Strands agents. This tool semantically retrieves data from Amazon Bedrock Knowledge Bases for RAG, memory, and other purposes. This tool requires a knowledge base ID, which will be provided through the environment variable `KNOWLEDGE_BASE_ID` defined below.

In [ ]:
# Find knowledge base
response = bedrock_agent_client.list_knowledge_bases()

# Search through knowledge bases
ncbi_kb_id = None
for kb in response['knowledgeBaseSummaries']:
    kb_name = kb['name']
    if 'ncbiKnowledgebase' in kb_name:
        ncbi_kb_id = kb['knowledgeBaseId']
        break

if ncbi_kb_id:
    print(f"Found Knowledge Base ID: {ncbi_kb_id}")
    os.environ["KNOWLEDGE_BASE_ID"] = ncbi_kb_id
    print("Knowledge Base will be integrated using direct Strands tool approach")
else:
    print("Warning: Knowledge Base not found. Internal evidence retrieval may not work.")
    ncbi_kb_id = None

# Create Strands Agent
This section creates an agent using the Strands framework

#### Define agent configuration and instructions

In [ ]:
clinical_research_agent_name = "Clinical-evidence-researcher-strands"
clinical_research_agent_description = "Research internal and external evidence using Strands framework"
clinical_research_agent_instruction = """You are a medical research assistant AI specialized in summarizing internal and external evidence related to HER2 biomarkers. 
Your primary tasks are to interpret user queries, gather internal and external evidence, and provide relevant medical insights based on those results. 
Use only the appropriate tools needed for specific questions. When searching for internal evidence, always use the knowledge base retrieval tool first. Please follow these guidelines carefully: 

1. When using the retrieve tool: 
   a. For internal evidence, leverage the knowledge base to retrieve relevant information. 
   b. Always include citations for specific content pieces (e.g., s3://bucket_name/test.pdf) in your response. Utilize the location information from the retrieve tool's response.

2. When querying PubMed:
   a. Summarize the results of each relevant study and cite the specific PubMed web link for that study.
   b. The JSON output includes 'link', 'title', and 'summary'. 
   c. Always include the title and link (e.g., 'https://pubmed.ncbi.nlm.nih.gov/') for each study in your response.  

4. When providing responses: 
   a. Begin by briefly summarizing your understanding of the user's query.  
   b. Explain the steps you are taking to resolve the query. Request additional clarification from the user if needed.  
   c. Distinguish between responses generated from internal evidence (knowledge base) and external evidence (PubMed API).  
   d. Conclude with a concise summary of your findings and potential implications for medical research.
"""

#### Define tools for Strands agent
We will use a custom tool to query PubMed and combine it with the retrieve tool from the Strands framework. The retrieve tool does not need to be defined as a function like custom tools; simply add it to the tool list.

In [ ]:
# Define tools using Strands @tool decorator
@tool
def query_pubmed(query: str) -> str:
    """
    Searches PubMed for relevant biomedical literature based on the user's query.
    This tool retrieves PubMed abstracts and returns relevant studies with title, link, and summary.
    
    Args:
        query (str): The search query for PubMed
    
    Returns:
        str: JSON string containing PubMed search results (title, link, summary)
    """
    
    pubmed = PubMed()

    print(f"\nPubMed query: {query}\n")
    result = pubmed.run(query)
    print(f"\nPubMed result: {result}\n")
    return result

# Create custom tool list
clinical_research_agent_tools = [query_pubmed, retrieve]
print(f"Created {len(clinical_research_agent_tools)} custom tools for Strands agent")

#### Set up AWS Bedrock provider for Strands

In [ ]:
# Strands용 베드락 모델 생성
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    region_name=region,
    temperature=0.1,
    streaming=False
)

#### Create Strands agent

In [ ]:
# Create Strands agent
try:
    # Using custom tools
    clinical_evidence_agent = Agent(
        model=model,
        tools=clinical_research_agent_tools,
        system_prompt=clinical_research_agent_instruction
    )
    
    print(f"Successfully created Strands agent: {clinical_research_agent_name}")
    print(f"Agent has {len(clinical_research_agent_tools)} tools available:")
    for tool in clinical_research_agent_tools:
        print(f"  - {tool.__name__}")
    
except Exception as e:
    print(f"Error creating agent: {e}")
    raise

#### Test Strands agent

In [ ]:
# Test agent with a research query
test_query = "Can you search for evidence on the efficacy of HER2-targeted therapies in HER2-positive breast cancer from both the internal knowledge base and PubMed?"

print(f"Testing agent with query: {test_query}")
print("=" * 140)

try:
    # Execute agent
    response = clinical_evidence_agent(test_query)
    
except Exception as e:
    print(f"Error during agent execution: {e}")
    import traceback
    traceback.print_exc()

#### Advanced usage examples

In [ ]:
# Examples of more complex queries
complex_queries = [
    "Search for evidence on trastuzumab resistance mechanisms in HER2-positive breast cancer",
    "Find studies on the efficacy of HER2-targeted therapy combined with immunotherapy",
    "What does the internal knowledge base say about HER2 biomarkers and treatment response prediction?"
]

def test_complex_query(query: str):
    """
    Test a complex query with the agent
    """
    print(f"\nTesting query: {query}")
    print("-" * 100)
    
    try:
        response = clinical_evidence_agent(query)
    except Exception as e:
        print(f"Error: {e}")

for query in complex_queries: 
    test_complex_query(query)

## Summary
This notebook demonstrates how to build an agent using the Strands framework that connects to Bedrock Knowledge Bases and the PubMed API.

### Available Tools:
- `query_pubmed`: Search medical literature from PubMed
- `KnowledgeBase`: Search internal evidence from NCBI knowledge base

### Research Capabilities:
- **Medical Literature Search** via PubMed API integration
- **Internal Evidence Retrieval** from curated knowledge bases
- **Comprehensive Research Synthesis** integrating multiple sources
- **Citation Tracking** with links to original studies
- **Biomarker Research** specialized for HER2-positive breast cancer research